# 13 — Kiểm chứng: GRU4Rec bản chính thức trên validation

Mục đích: đối chiếu GRU4Rec **tự cài** của dự án (notebook 07, 09) với **code chính thức** của tác giả ([hidasib/GRU4Rec_PyTorch_Official](https://github.com/hidasib/GRU4Rec_PyTorch_Official), commit `d1fc311`) trên **cùng bộ chia, cùng query validation, cùng bộ lọc ứng viên, fallback và metric**. Nếu hai bản cho NDCG@20 cùng bậc, đó là bằng chứng code của dự án không lệch khỏi mô hình gốc. Người dùng duyệt ngày 26/11/2026.

- **Chỉ train và validation; không đọc test.** Notebook này không thay đổi kết quả chính của notebook 10.
- **Code chính thức không nằm trong repo** (license: miễn phí cho nghiên cứu và giáo dục). Notebook tải đúng commit đã ghim vào `external/` (đã gitignore) và kiểm tra SHA-256 trước khi import. Không sửa file nào của họ.
- **Cấu hình:** giữ nguyên `paramfiles/retailrocket_bprmax_shared_best.py` của họ: BPR-max, constrained embedding, 224 hidden, batch 80, Adagrad có momentum (lr 0,05, momentum 0,4), dropout 0,5 / 0,05, `n_sample` 2048, `sample_alpha` 0,4, `bpreg` 1,95, 10 epoch. Chấm NDCG@20 validation sau mỗi epoch; báo cáo cả epoch 10 (đúng bản gốc) và epoch tốt nhất (thủ tục dừng sớm của dự án).

Các điểm lệch so với giao thức chính, đều chỉ áp dụng cho notebook này:

| Điểm | Bản chính thức cần | Cách làm ở đây |
| --- | --- | --- |
| Đầu vào | Một sự kiện mỗi bước, dự đoán sự kiện kế tiếp | Mỗi sự kiện train là một dòng `SessionId, ItemId, Time`, đủ mọi hành vi, phiên 30 phút |
| Sự kiện cùng timestamp | Phải có thứ tự | Xếp theo `event_id` (người dùng chốt 26/11/2026). Giao thức chính cấm thứ tự giả này; ở đây nó là điều kiện bắt buộc để chạy bản chính thức |
| Item ngoài vocabulary trong prefix | Bị bỏ khỏi chuỗi | Bỏ như bản gốc (GRU4Rec của dự án dùng token UNK); prefix không còn item nào → trạng thái ẩn 0 |
| Phiên dài (D3) | Không cắt | Không cắt (dự án cắt cửa sổ 51 nhóm khi train) |
| NumPy 2 | Code thử với NumPy 1.19 | `_init_numpy_weights` được thay bằng bản cho đúng giá trị NumPy 1 (`src/rr/official_gru.py`) |

Đánh giá: với mỗi query validation, trạng thái ẩn sau sự kiện cuối của prefix → điểm toàn bộ item (`state · Wy + By`, đúng cách bản gốc chấm) → 200 ứng viên đầu → bộ lọc ứng viên tại t, fallback Popularity, NDCG@20 của dự án. Bổ sung: Recall@20 / MRR@20 theo `evaluation.batch_eval` của bản gốc (dự đoán sự kiện kế tiếp trên chuỗi sự kiện validation), chỉ để tham khảo.

Đầu ra: `models/gru4rec_official/{config.json, model.pt}` (không commit `.pt`), `reports/13_gru4rec_official/metrics.json`.

## 1. Cấu hình, code chính thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import copy, os, subprocess, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, report_path, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.evaluation import run
from rr.metrics import paired_bootstrap
from rr.models import popularity
from rr.official_gru import (OFFICIAL_COMMIT, OFFICIAL_SHA256, OFFICIAL_URL, OfficialTrainer, load_official, lf_sha256,
                             query_states, read_paramfile, to_official_frame, topn_items)

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
require(PROTOCOL["version"] == "1.3.2", "Notebook này viết cho protocol v1.3.2.")
PROCESSED = ROOT / "data/processed"
OUT = ROOT / "reports/13_gru4rec_official"
MODEL_DIR = ROOT / "models/gru4rec_official"
CODE_DIR = ROOT / "external/GRU4Rec_PyTorch_Official"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
SEED = PROTOCOL["seed"]
PAIRED = PROTOCOL["evaluation"]["paired_bootstrap"]
WIDTH = 200
TCOLS = ["query_id", "itemid", "target_is_repeat"]
torch.set_num_threads(4)

LOG = os.environ.get("RR_PROGRESS_LOG")
def log(msg):
    line = f"{pd.Timestamp.now(tz='UTC'):%H:%M:%S} [13] {msg}"
    print(line, flush=True)
    if LOG:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")

if not (CODE_DIR / "gru4rec_pytorch.py").exists():
    CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-q", OFFICIAL_URL, str(CODE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "-q", OFFICIAL_COMMIT], check=True)
gm, ge = load_official(CODE_DIR)          # checks SHA-256 of gru4rec_pytorch.py and evaluation.py
PARAMS = read_paramfile(CODE_DIR / "paramfiles/retailrocket_bprmax_shared_best.py")   # read as a literal, not executed
display(pd.Series(PARAMS, name="retailrocket_bprmax_shared_best"))

prev = {n: read_json(report_path(ROOT, n)) for n in ["session_split", "metadata", "samples"]}
val09 = read_json(ROOT / "reports/09_validation_30_60/metrics.json")
reh = read_json(ROOT / "reports/10_final_rehearsal/metrics.json")
require(val09["protocol"]["version"] == reh["protocol"]["version"] == PROTOCOL["version"], "Report 09/10 được tạo với protocol khác.")
require(reh["rehearsal"] and reh["status"] == "PASS", "Cần bản diễn tập notebook 10 đã PASS.")
expected = {"data/processed/events_sessionized.parquet": prev["session_split"]["artifacts"]["data/processed/events_sessionized.parquet"]}
for p in ["item_property_history", "item_catalog"]:
    expected[f"data/processed/{p}.parquet"] = prev["metadata"]["artifacts"][f"data/processed/{p}.parquet"]
for p in ["item_vocab.parquet", "samples/validation_queries.parquet", "samples/validation_targets.parquet"]:
    expected[f"data/processed/{p}"] = prev["samples"]["artifacts"][f"data/processed/{p}"]
expected[reh["per_query"]["path"]] = reh["per_query"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
for p in ["configs/protocol.json", "reports/09_validation_30_60/metrics.json", "reports/10_final_rehearsal/metrics.json"]:
    inputs[p] = fingerprint(ROOT / p)
official_code = {"url": OFFICIAL_URL, "commit": OFFICIAL_COMMIT, "sha256_lf": OFFICIAL_SHA256,
                 "paramfile": "paramfiles/retailrocket_bprmax_shared_best.py",
                 "paramfile_sha256_lf": lf_sha256(CODE_DIR / "paramfiles/retailrocket_bprmax_shared_best.py")}
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| torch", torch.__version__, "| threads", torch.get_num_threads())
log("bắt đầu")

## 2. Ví dụ biên trên dữ liệu nhỏ

- Sự kiện cùng timestamp được xếp theo `event_id`.
- Bản vá NumPy 2 cho giá trị `float32` đúng như NumPy 1.
- Vòng lặp từng epoch của notebook cho **trọng số trùng tuyệt đối** với `GRU4Rec.fit` gốc (cùng seed).
- `query_states` (chạy nhiều phiên song song) cho cùng trạng thái ẩn với cách đưa từng sự kiện của một prefix qua mô hình; item ngoài vocabulary bị bỏ qua.

Các trường hợp này cũng có trong `tests/test_official_gru.py`.

In [ ]:
_ev = pd.DataFrame({"event_id": [5, 3, 4], "session_id": [1, 1, 1], "group_index": [1, 1, 2], "itemid": [50, 30, 40], "timestamp": [10, 10, 20]})
assert to_official_frame(_ev)["ItemId"].tolist() == [30, 50, 40]

np.random.seed(0)
_w = gm.GRU4RecModel._init_numpy_weights(None, (300, 224))
_s = np.float32(np.sqrt(6.0 / 524))
np.random.seed(0)
assert _w.dtype == np.float32 and np.array_equal(_w, np.random.rand(300, 224).astype("float32") * np.float32(2) * _s - _s)

_rows, _rng = [], np.random.default_rng(0)
for s in range(1, 13):
    for t in range(1, 5):
        for _ in range(1 + (t == 2)):
            _rows.append((len(_rows), s, t, int(_rng.integers(10, 20)), 100 * s + t))
_toy = pd.DataFrame(_rows, columns=["event_id", "session_id", "group_index", "itemid", "timestamp"])
_p = {**PARAMS, "layers": [8], "batch_size": 4, "n_sample": 4, "n_epochs": 1}
_ours = OfficialTrainer(gm, _p, to_official_frame(_toy), seed=7, sample_cache_max_size=400)
_ours.run_epoch()
torch.manual_seed(7); np.random.seed(7)
_ref = gm.GRU4Rec(**_p, device=torch.device("cpu"))
_ref.fit(to_official_frame(_toy), sample_cache_max_size=400)
for _a, _b in zip(_ours.gru.model.state_dict().values(), _ref.model.state_dict().values()):
    assert torch.equal(_a, _b)

_val = pd.DataFrame({"event_id": [100, 102, 101, 103], "session_id": [50, 50, 50, 50], "group_index": [1, 2, 2, 3],
                     "itemid": [11, 999, 12, 13], "timestamp": [1, 2, 2, 3]})
_st = query_states(_ours.gru, _val, pd.DataFrame({"session_id": [50, 50], "prefix_len_groups": [2, 3]}))
_H = [torch.zeros((1, 8))]
with torch.no_grad():
    for _i in [11, 12]:
        _E, _, _ = _ours.gru.model.embed(torch.tensor([_ours.gru.data_iterator.itemidmap[_i]]), _H, None)
        _ours.gru.model.hidden_step(_E, _H, training=False)
torch.testing.assert_close(_st[0], _H[0][0])
del _ev, _w, _s, _rows, _rng, _toy, _p, _ours, _ref, _a, _b, _val, _st, _H, _E, _i
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu

Train = mọi sự kiện thuộc `train` của phiên 30 phút (notebook 02), một sự kiện mỗi dòng. Item map của bản chính thức phải trùng vocabulary train của dự án, nên hai bản chấm trên cùng tập item.

In [ ]:
ev = pd.read_parquet(PROCESSED / "events_sessionized.parquet", columns=["event_id", "session_id", "group_index", "itemid", "event", "timestamp", "split"])
ev["split"] = ev["split"].astype(str)
train_ev = ev[ev["split"] == "train"].reset_index(drop=True)
val_ev = ev[ev["split"] == "validation"].reset_index(drop=True)
del ev
train = to_official_frame(train_ev)
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
val_q = pd.read_parquet(PROCESSED / "samples/validation_queries.parquet")
val_t = pd.read_parquet(PROCESSED / "samples/validation_targets.parquet", columns=TCOLS)
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
fb_weights = val09["selected"]["30"]["popularity"]["fallback_weights"]
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(train_ev, fb_weights))
require(train_ev["timestamp"].max() < val_q["query_time_ms"].min(), "Train phải nằm trước validation.")
DATA = {"train_events": int(len(train)), "train_sessions": int(train["SessionId"].nunique()),
        "train_sessions_ge2_events": int((train.groupby("SessionId").size() >= 2).sum()),
        "events_sharing_timestamp_in_session": int(train.duplicated(["SessionId", "Time"], keep=False).sum()),
        "validation_events": int(len(val_ev)), "validation_queries": int(len(val_q)), "fallback_weights": fb_weights}
display(pd.Series(DATA))

## 4. Huấn luyện bản chính thức và chấm validation sau mỗi epoch

10 epoch đúng paramfile. Sau mỗi epoch: trạng thái ẩn của 92.566 query validation → Top-200 → bộ lọc, fallback, NDCG@20 của dự án. Trọng số của epoch tốt nhất được giữ lại.

In [ ]:
t_all = time.perf_counter()
trainer = OfficialTrainer(gm, PARAMS, train, seed=SEED)
gru = trainer.gru
require(set(gru.data_iterator.itemidmap.index) == set(vocab["itemid"]), "Item map của bản chính thức khác vocabulary train.")

def evaluate_official(gru):
    t0 = time.perf_counter()
    states = query_states(gru, val_ev, val_q)
    items, scores = topn_items(gru, states, WIDTH)
    report, per_query, *_ = run(items, scores, val_q, val_t, flt, fb_items, fb_scores)
    report["query_seconds"] = round(time.perf_counter() - t0, 1)
    return report, per_query

history_rows, reports, per_query, best = [], {}, {}, None
for epoch in range(1, PARAMS["n_epochs"] + 1):
    row = trainer.run_epoch()
    reports[epoch], per_query[epoch] = evaluate_official(gru)
    row["ndcg@20"] = reports[epoch]["all"]["ndcg@20"]
    history_rows.append(row)
    if best is None or row["ndcg@20"] > reports[best]["all"]["ndcg@20"]:
        best, best_state = epoch, copy.deepcopy(gru.model.state_dict())
    log(f"epoch {epoch}: loss={row['loss']:.4f} ({row['train_seconds']:.0f}s), ndcg@20={row['ndcg@20']:.4f} ({reports[epoch]['query_seconds']:.0f}s)")
LAST = PARAMS["n_epochs"]
last_state = copy.deepcopy(gru.model.state_dict())
gru.model.load_state_dict(best_state)
require(abs(evaluate_official(gru)[0]["all"]["ndcg@20"] - reports[best]["all"]["ndcg@20"]) < 1e-12, "Không khôi phục được epoch tốt nhất.")
display(pd.DataFrame(history_rows).set_index("epoch"))
log(f"huấn luyện xong ({(time.perf_counter() - t_all) / 60:.0f} phút): epoch tốt nhất {best}, ndcg@20={reports[best]['all']['ndcg@20']:.4f}; epoch {LAST}: {reports[LAST]['all']['ndcg@20']:.4f}")

## 5. So với GRU4Rec của dự án trên cùng query validation

- **Bảng 1:** HR / MRR / NDCG@20, toàn bộ và tách target mới / lặp. Số của GRU4Rec dự án lấy từ notebook 09 (phiên 30 phút).
- **Bảng 2:** paired bootstrap NDCG@20, bản chính thức − bản của dự án, trên cùng thứ tự query (điểm từng query của dự án lấy từ bản diễn tập notebook 10, biến thể `train_only`, `q30`).
- **Bổ sung:** Recall@20 / MRR@20 theo `evaluation.batch_eval` của bản gốc (chế độ `conservative`) trên chuỗi sự kiện validation, cho epoch tốt nhất. Đây là bài toán dự đoán sự kiện kế tiếp, khác bài toán của dự án; chỉ để đặt cạnh số tác giả công bố (Recall@20 0,5186, MRR@20 0,2175 trên bộ chia riêng của họ), không so trực tiếp.

In [ ]:
proj = {n: val09["gru4rec"]["30"][n]["report"] for n in ["gru4rec_item", "gru4rec"]}
require(abs(proj["gru4rec_item"]["all"]["ndcg@20"] - val09["validation_table"]["rows"]["30"]["gru4rec_item"]["ndcg@20"]) < 1e-12, "Số notebook 09 không nhất quán.")
ROWS = {f"official_epoch{best}_best": reports[best], f"official_epoch{LAST}": reports[LAST],
        "project_gru4rec_item": proj["gru4rec_item"], "project_gru4rec": proj["gru4rec"]}
table = pd.DataFrame({n: {**{f"{part}:{m}": r[part][m] for part in ["all"] for m in ["hit@20", "mrr@20", "ndcg@20"]},
                          "new:ndcg@20": r["new_targets"]["ndcg@20"], "repeat:ndcg@20": r["repeat_targets"]["ndcg@20"]}
                      for n, r in ROWS.items()}).T
display(table.style.format("{:.4f}"))

pq_reh = pd.read_parquet(ROOT / reh["per_query"]["path"])
pq_reh = pq_reh[(pq_reh["fit_threshold"] == "30") & (pq_reh["variant"] == "train_only") & (pq_reh["eval_set"] == "q30")]
PAIRS = {}
for ep in sorted({best, LAST}):
    ours = per_query[ep].set_index("query_id")["ndcg@20"]
    for m in ["gru4rec_item", "gru4rec"]:
        theirs = pq_reh[pq_reh["model"] == m].set_index("query_id")["ndcg@20"]
        require(theirs.index.equals(ours.index), f"{m}: thứ tự query khác.")
        require(abs(theirs.mean() - proj[m]["all"]["ndcg@20"]) < 1e-12, f"{m}: điểm từng query không khớp notebook 09.")
        PAIRS[f"official_epoch{ep} - project_{m}"] = paired_bootstrap(ours.to_numpy(), theirs.to_numpy(), PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])
display(pd.DataFrame(PAIRS).T[["mean_diff", "low", "high", "p_not_better"]])

t0 = time.perf_counter()
recall, mrr = ge.batch_eval(gru, to_official_frame(val_ev), cutoff=[20], batch_size=512, mode="conservative")
OFFICIAL_EVAL = {"epoch": best, "mode": "conservative", "recall@20": float(recall[20]), "mrr@20": float(mrr[20]),
                 "seconds": round(time.perf_counter() - t0, 1), "data": "validation events, 30-minute sessions, items outside the train item map dropped"}
display(pd.Series(OFFICIAL_EVAL))
log(f"so sánh xong: official best - project gru4rec_item = {PAIRS[f'official_epoch{best} - project_gru4rec_item']['mean_diff']:+.4f}")

## 6. Lưu mô hình và báo cáo

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save(best_state, MODEL_DIR / "model.pt")
config = {"name": "gru4rec_official", "params": PARAMS, "best_epoch": best, "epochs_trained": LAST, "seed": SEED,
          "n_items": int(gru.data_iterator.n_items), "official_code": official_code,
          "input": "train events of 30-minute sessions, one row per event, same-timestamp events ordered by event_id"}
write_json(MODEL_DIR / "config.json", config)
artifacts = {rel(MODEL_DIR / f, ROOT): fingerprint(MODEL_DIR / f) for f in ["model.pt", "config.json"]}
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
require(lf_sha256(CODE_DIR / "gru4rec_pytorch.py") == OFFICIAL_SHA256["gru4rec_pytorch.py"], "Code chính thức thay đổi trong lúc chạy.")
write_json(OUT / "metrics.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
    "notebook": "notebooks/13_gru4rec_chinh_thuc_validation.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "scope": "verification on validation only; test not read; main results of notebook 10 unchanged",
    "approved": "user approved on 2026-11-26: official code, its RetailRocket paramfile, same-timestamp events ordered by event_id",
    "official_code": official_code,
    "adaptations": {"same_timestamp_order": "event_id (fake order, forbidden in the main protocol, accepted for this run only)",
                    "oov_in_prefix": "dropped as in the official iterator; prefix without known items -> zero state",
                    "long_sessions": "not cut (the project cuts 51-group windows when training)",
                    "numpy2": "GRU4RecModel._init_numpy_weights replaced by the NumPy-1-equivalent float32 computation"},
    "inputs": inputs, "artifacts": artifacts, "data": DATA, "params": PARAMS,
    "history": history_rows, "reports_by_epoch": {str(e): r for e, r in reports.items()}, "best_epoch": best,
    "comparison": {"rows": {n: {"all": r["all"], "new_targets": r["new_targets"], "repeat_targets": r["repeat_targets"]} for n, r in ROWS.items()},
                   "paired_ndcg@20": PAIRS},
    "official_batch_eval": OFFICIAL_EVAL,
    "status": "PASS"})
log("xong, báo cáo: " + rel(OUT / "metrics.json", ROOT))